In [1]:
"""
WSOD with Dedicated Spatial Attention Module
Freezes backbone, trains only spatial attention layer + FC
"""

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader
import torchvision.transforms as transforms
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from tqdm import tqdm
from sklearn.metrics import roc_auc_score, precision_score, recall_score, f1_score
import timm
import cv2

# Device configuration
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

c:\Users\Edrill-LT\Documents\Projects\Python\Thoracic-Disease-Classifier-ResNet50\.torch\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Using device: cuda


In [2]:
# ============================================================================
# Spatial Attention Module
# ============================================================================

class SpatialAttentionModule(nn.Module):
    """
    Learnable spatial attention module
    Takes features from backbone and outputs spatial attention weights
    """
    def __init__(self, in_channels=2048, reduction=8):
        super(SpatialAttentionModule, self).__init__()
        
        # Spatial attention pathway
        self.conv1 = nn.Conv2d(in_channels, in_channels // reduction, kernel_size=1)
        self.bn1 = nn.BatchNorm2d(in_channels // reduction)
        self.relu = nn.ReLU(inplace=True)
        
        self.conv2 = nn.Conv2d(in_channels // reduction, in_channels // reduction, 
                               kernel_size=3, padding=1)
        self.bn2 = nn.BatchNorm2d(in_channels // reduction)
        
        # Output attention map
        self.conv_out = nn.Conv2d(in_channels // reduction, 1, kernel_size=1)
        self.sigmoid = nn.Sigmoid()
        
    def forward(self, x):
        """
        Args:
            x: [B, 2048, 7, 7] features from backbone
        Returns:
            attention: [B, 1, 7, 7] spatial attention map
            weighted_features: [B, 2048, 7, 7] attention-weighted features
        """
        # Generate spatial attention
        att = self.conv1(x)
        att = self.bn1(att)
        att = self.relu(att)
        
        att = self.conv2(att)
        att = self.bn2(att)
        att = self.relu(att)
        
        att = self.conv_out(att)
        attention = self.sigmoid(att)  # [B, 1, 7, 7]
        
        # Apply attention to features
        weighted_features = x * attention
        
        return attention, weighted_features

In [3]:
# ============================================================================
# WSOD Model with Spatial Attention
# ============================================================================

class WSODModelWithAttention(nn.Module):
    def __init__(self, base_model, num_classes=2, freeze_backbone=True):
        super(WSODModelWithAttention, self).__init__()
        
        # Freeze backbone if requested
        if freeze_backbone:
            for param in base_model.parameters():
                param.requires_grad = False
            print("✓ Backbone frozen")
        
        # Extract feature extractor (everything except FC)
        self.features = nn.Sequential(*list(base_model.children())[:-2])  # Up to layer4
        self.avgpool = list(base_model.children())[-2]  # AdaptiveAvgPool2d
        
        # Add spatial attention module (TRAINABLE)
        self.spatial_attention = SpatialAttentionModule(in_channels=2048)
        
        # Classifier head (TRAINABLE)
        self.fc = nn.Linear(2048, num_classes)
        
        # If backbone is frozen, unfreeze only the FC from base_model
        if freeze_backbone:
            # Copy weights from original FC if available
            original_fc = list(base_model.children())[-1]
            if isinstance(original_fc, nn.Linear):
                self.fc.weight.data.copy_(original_fc.weight.data)
                self.fc.bias.data.copy_(original_fc.bias.data)
        
    def forward(self, x, return_attention=False):
        # Extract features from frozen backbone
        features = self.features(x)  # [B, 2048, 7, 7]
        
        # Apply spatial attention
        attention_map, weighted_features = self.spatial_attention(features)
        
        # Global average pooling
        pooled = self.avgpool(weighted_features)  # [B, 2048, 1, 1]
        pooled = pooled.flatten(1)  # [B, 2048]
        
        # Classification
        output = self.fc(pooled)
        
        if return_attention:
            return output, attention_map
        return output
    
    def get_attention_map(self, x):
        """Get spatial attention map for visualization"""
        with torch.no_grad():
            features = self.features(x)
            attention_map, _ = self.spatial_attention(features)
        return attention_map

In [4]:
# ============================================================================
# CONFIGURATION
# ============================================================================

BATCH_SIZE = 16
NUM_EPOCHS = 30  # More epochs since attention module learns from scratch
LEARNING_RATE = 3e-4  # Higher LR for attention module (backbone is frozen)
IMAGE_SIZE = 224
NUM_WORKERS = 4
WEIGHT_DECAY = 0.01
ATTENTION_WEIGHT = 1.0  # Can be higher since backbone is safe
SIGMA_SCALE = 1.0  # Start strict, can adjust if needed

# Paths
split_base_path = "./dataset_nodule21/cxr_images/proccessed_data/split_data"

test_images_path = f"{split_base_path}/test/images"

test_csv_path = f"{split_base_path}/test/metadata_test.csv"

In [5]:
# ============================================================================
# LOAD DATA
# ============================================================================

print("\nLoading datasets...")
test_df = pd.read_csv(test_csv_path)

print(f"Test samples: {len(test_df)}")

val_transform = transforms.Compose([
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

# Create datasets with bbox
from NoduleDS import NoduleDataset
test_dataset = NoduleDataset(test_df, test_images_path, transform=val_transform, return_bbox=True)

test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True)



Loading datasets...
Test samples: 785


In [6]:
# ============================================================================
# IoU CALCULATION UTILITIES
# ============================================================================

def calculate_iou(attention_map1, attention_map2, threshold=0.5):
    """
    Calculate IoU between two attention maps
    
    Args:
        attention_map1: [B, 1, H, W] or [B, H, W]
        attention_map2: [B, 1, H, W] or [B, H, W]
        threshold: threshold for binarization
    
    Returns:
        iou_scores: [B] IoU score for each sample in batch
    """
    # Ensure same shape
    if attention_map1.dim() == 4:
        attention_map1 = attention_map1.squeeze(1)
    if attention_map2.dim() == 4:
        attention_map2 = attention_map2.squeeze(1)
    
    # Resize to same size if needed
    if attention_map1.shape != attention_map2.shape:
        attention_map2 = F.interpolate(
            attention_map2.unsqueeze(1), 
            size=attention_map1.shape[-2:], 
            mode='bilinear', 
            align_corners=False
        ).squeeze(1)
    
    # Binarize attention maps
    binary_map1 = (attention_map1 > threshold).float()
    binary_map2 = (attention_map2 > threshold).float()
    
    # Calculate IoU for each sample in batch
    batch_size = binary_map1.size(0)
    iou_scores = []
    
    for i in range(batch_size):
        map1 = binary_map1[i]
        map2 = binary_map2[i]
        
        # Intersection and Union
        intersection = (map1 * map2).sum()
        union = (map1 + map2).clamp(0, 1).sum()
        
        # Avoid division by zero
        if union > 0:
            iou = intersection / union
        else:
            iou = torch.tensor(0.0)
        
        iou_scores.append(iou.item())
    
    return np.array(iou_scores)

def upscale_attention_map(attention_map, target_size=(224, 224)):
    """
    Upscale attention map to target size for visualization
    
    Args:
        attention_map: [B, 1, H, W] tensor
        target_size: (H, W) tuple
    
    Returns:
        upscaled_map: [B, 1, H, W] tensor at target size
    """
    return F.interpolate(
        attention_map,
        size=target_size,
        mode='bilinear',
        align_corners=False
    )

In [7]:
# ============================================================================
# MAIN COMPARISON FUNCTION
# ============================================================================

def compare_attention_maps(custom_model_path, baseline_model_path, 
                          test_loader, device, threshold=0.5):
    """
    Compare attention maps between custom and baseline models
    
    Args:
        custom_model_path: path to custom model checkpoint
        baseline_model_path: path to baseline model checkpoint
        test_loader: DataLoader for test set
        device: torch device
        threshold: threshold for binarization
    
    Returns:
        results: dict with IoU scores and statistics
    """
    print("\n" + "="*80)
    print("LOADING MODELS")
    print("="*80)
    
    # Load custom model
    base_model = timm.create_model('resnet50', pretrained=True, num_classes=2)
    custom_model = WSODModelWithAttention(base_model, num_classes=2, freeze_backbone=True).to(device)
    custom_checkpoint = torch.load(custom_model_path, map_location=device)
    custom_model.load_state_dict(custom_checkpoint['model_state_dict'])
    custom_model.eval()
    print(f"✓ Loaded custom model from {custom_model_path}")
    print(f"  Epoch: {custom_checkpoint.get('epoch', 'unknown')}")
    
    # Load baseline model
    baseline_model = BaselineResNet50(num_classes=2).to(device)
    baseline_checkpoint = torch.load(baseline_model_path, map_location=device)
    baseline_model.load_state_dict(baseline_checkpoint['model_state_dict'])
    baseline_model.eval()
    print(f"✓ Loaded baseline model from {baseline_model_path}")
    print(f"  Epoch: {baseline_checkpoint.get('epoch', 'unknown')}")
    
    # Storage for IoU scores
    all_iou_scores = []
    all_labels = []
    
    print("\n" + "="*80)
    print("COMPUTING IoU SCORES ON TEST SET")
    print("="*80)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Processing batches")):
            # Unpack batch
            if len(batch) == 3:
                images, labels, _ = batch  # With bboxes
            else:
                images, labels = batch
            
            images = images.to(device)
            labels = labels.to(device)
            
            # Get attention maps from both models
            custom_attention = custom_model.get_attention_map(images)  # [B, 1, 7, 7]
            baseline_attention = baseline_model.get_attention_map(images)  # [B, 1, 7, 7]
            
            # Calculate IoU for this batch
            batch_iou = calculate_iou(custom_attention, baseline_attention, threshold)
            
            # Store results
            all_iou_scores.extend(batch_iou)
            all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays
    all_iou_scores = np.array(all_iou_scores)
    all_labels = np.array(all_labels)
    
    # Calculate statistics
    results = {
        'all_iou_scores': all_iou_scores,
        'all_labels': all_labels,
        'mean_iou': np.mean(all_iou_scores),
        'std_iou': np.std(all_iou_scores),
        'median_iou': np.median(all_iou_scores),
        'min_iou': np.min(all_iou_scores),
        'max_iou': np.max(all_iou_scores),
        'threshold': threshold
    }
    
    # Calculate per-class IoU
    for label in np.unique(all_labels):
        mask = all_labels == label
        class_iou = all_iou_scores[mask]
        label_name = "Nodule" if label == 1 else "Normal"
        results[f'mean_iou_class_{label_name}'] = np.mean(class_iou)
        results[f'std_iou_class_{label_name}'] = np.std(class_iou)
        results[f'count_class_{label_name}'] = len(class_iou)
    
    return results, custom_model, baseline_model

In [8]:
# ============================================================================
# MAIN COMPARISON FUNCTION
# ============================================================================

def compare_attention_maps(custom_model_path, baseline_model_path, 
                          test_loader, device, threshold=0.5):
    """
    Compare attention maps between custom and baseline models
    
    Args:
        custom_model_path: path to custom model checkpoint
        baseline_model_path: path to baseline model checkpoint
        test_loader: DataLoader for test set
        device: torch device
        threshold: threshold for binarization
    
    Returns:
        results: dict with IoU scores and statistics
    """
    print("\n" + "="*80)
    print("LOADING MODELS")
    print("="*80)
    
    # Load custom model
    base_model = timm.create_model('resnet50', pretrained=True, num_classes=2)
    custom_model = WSODModelWithAttention(base_model, num_classes=2, freeze_backbone=True).to(device)
    custom_checkpoint = torch.load(custom_model_path, map_location=device)
    custom_model.load_state_dict(custom_checkpoint['model_state_dict'])
    custom_model.eval()
    print(f"✓ Loaded custom model from {custom_model_path}")
    print(f"  Epoch: {custom_checkpoint.get('epoch', 'unknown')}")
    
    
    # Load baseline model
    baseline_model = timm.create_model('resnet50', pretrained=False, num_classes=2).to(device)
    baseline_checkpoint = torch.load(baseline_model_path, map_location=device)
    baseline_model.load_state_dict(baseline_checkpoint['model_state_dict'])
    baseline_model.eval()
    print(f"✓ Loaded baseline model from {baseline_model_path}")
    print(f"  Epoch: {baseline_checkpoint.get('epoch', 'unknown')}")
    
    # Storage for IoU scores
    all_iou_scores = []
    all_labels = []
    
    print("\n" + "="*80)
    print("COMPUTING IoU SCORES ON TEST SET")
    print("="*80)
    
    with torch.no_grad():
        for batch_idx, batch in enumerate(tqdm(test_loader, desc="Processing batches")):
            # Unpack batch
            if len(batch) == 3:
                images, labels, _ = batch  # With bboxes
            else:
                images, labels = batch
            
            images = images.to(device)
            labels = labels.to(device)
            
            # Get attention maps from both models
            custom_attention = custom_model.get_attention_map(images)  # [B, 1, 7, 7]
            baseline_attention = baseline_model.get_attention_map(images)  # [B, 1, 7, 7]
            
            # Calculate IoU for this batch
            batch_iou = calculate_iou(custom_attention, baseline_attention, threshold)
            
            # Store results
            all_iou_scores.extend(batch_iou)
            all_labels.extend(labels.cpu().numpy())
    
    # Convert to numpy arrays
    all_iou_scores = np.array(all_iou_scores)
    all_labels = np.array(all_labels)
    
    # Calculate statistics
    results = {
        'all_iou_scores': all_iou_scores,
        'all_labels': all_labels,
        'mean_iou': np.mean(all_iou_scores),
        'std_iou': np.std(all_iou_scores),
        'median_iou': np.median(all_iou_scores),
        'min_iou': np.min(all_iou_scores),
        'max_iou': np.max(all_iou_scores),
        'threshold': threshold
    }
    
    # Calculate per-class IoU
    for label in np.unique(all_labels):
        mask = all_labels == label
        class_iou = all_iou_scores[mask]
        label_name = "Nodule" if label == 1 else "Normal"
        results[f'mean_iou_class_{label_name}'] = np.mean(class_iou)
        results[f'std_iou_class_{label_name}'] = np.std(class_iou)
        results[f'count_class_{label_name}'] = len(class_iou)
    
    return results, custom_model, baseline_model

In [ ]:
# ============================================================================
# HELPER FUNCTION TO PRINT RESULTS
# ============================================================================

def print_iou_results(results):
    """Pretty print IoU comparison results"""
    print("\n" + "="*80)
    print("IoU COMPARISON RESULTS")
    print("="*80)
    print(f"Threshold: {results['threshold']}")
    print(f"\nOverall Statistics:")
    print(f"  Mean IoU:   {results['mean_iou']:.4f}")
    print(f"  Std IoU:    {results['std_iou']:.4f}")
    print(f"  Median IoU: {results['median_iou']:.4f}")
    print(f"  Min IoU:    {results['min_iou']:.4f}")
    print(f"  Max IoU:    {results['max_iou']:.4f}")
    
    print(f"\nPer-Class Statistics:")
    if 'mean_iou_class_Normal' in results:
        print(f"  Normal (n={results['count_class_Normal']}):")
        print(f"    Mean IoU: {results['mean_iou_class_Normal']:.4f}")
        print(f"    Std IoU:  {results['std_iou_class_Normal']:.4f}")
    
    if 'mean_iou_class_Nodule' in results:
        print(f"  Nodule (n={results['count_class_Nodule']}):")
        print(f"    Mean IoU: {results['mean_iou_class_Nodule']:.4f}")
        print(f"    Std IoU:  {results['std_iou_class_Nodule']:.4f}")
    
    print("="*80)

print_iou_results(